<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/ColabMD_Edu_Protein_NA_Run_AAMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Mount to GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Create the folder > ColabMD-Edu_Protein-NA
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-NA"

In [ ]:
import sys
sys.path.append('/usr/local/lib/python3.13/site-packages')

In [ ]:
# @title Change Directory > ColabMD-Edu_Protein-NA!pip install -q condacolab
%%bash
# Download the latest Miniforge installer
wget -q -O Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"

# Install it silently into Colab's main directory
bash Miniforge3.sh -b -f -p /usr/local

# Clean up the installer script
rm Miniforge3.sh

# Set conda-forge as the default channel
conda config --system --add channels conda-forge
conda config --system --set channel_priority strict

In [ ]:
# @title Force remove the hidden configuration pin file causing the glitch
!rm -f /usr/local/conda-meta/pinned

In [ ]:
# @title Install GROMACS via mamba cleanly (T4 GPU)
!mamba install -c conda-forge gromacs=*=*cuda* -y

In [ ]:
# @title Verify it is installed and using AVX2 & CUDA
!gmx --version

In [ ]:
# @title Change Directory > ColabMD-Edu_Protein-NA
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA')

In [ ]:
# @title Clone AAMD directory from GitHub
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-NA/MD/AAMD/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-NA

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d AAMD

**All-Atomic MD Simulation**

In [ ]:
# @title Change Directory > AAMD
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD')

In [ ]:
# @title OPTIONAL - Convert PDB format to gromacs format (gro) for RNA-Aptamer & Crystallized PDB
import os
import shutil
import subprocess
from pathlib import Path

# ==============================================================================
# 1. SETUP LOCAL FORCE FIELD DIRECTORY
# ==============================================================================
def find_gromacs_share():
    search_paths = [
        Path("/usr/local/share/gromacs/top"),
        Path("/usr/share/gromacs/top"),
        Path(os.environ.get("CONDA_PREFIX", "")) / "share/gromacs/top"
    ]
    for p in search_paths:
        if (p / "amber99sb-ildn.ff").exists():
            return p
    raise FileNotFoundError("Could not locate GROMACS top directory containing amber99sb-ildn.ff")

gmx_top = find_gromacs_share()
local_ff = Path("./amber99sb-ildn.ff")

if local_ff.exists():
    shutil.rmtree(local_ff)
shutil.copytree(gmx_top / "amber99sb-ildn.ff", local_ff)
print(f"-> Fresh copy of amber99sb-ildn.ff initialized at: {local_ff.resolve()}")

# Remove any existing local vdwradii.dat to use standard GROMACS default radii
if Path("./vdwradii.dat").exists():
    Path("./vdwradii.dat").unlink()

# ==============================================================================
# 2. CONFIGURE residuetypes.dat (STRICTLY 2 COLUMNS PER LINE)
# ==============================================================================
local_res_file = Path("./residuetypes.dat")
clean_lines = []

with open(gmx_top / "residuetypes.dat", "r") as f:
    for line in f:
        tokens = line.strip().split()
        if len(tokens) == 2 and not tokens[0].startswith(("#", ";")):
            clean_lines.append(f"{tokens[0]:<6} {tokens[1]}\n")

new_entries = [
    ("CFZ", "RNA"), ("CFZ5", "RNA"), ("CFZ3", "RNA"), ("CFZN", "RNA"),
    ("UFT", "RNA"), ("UFT5", "RNA"), ("UFT3", "RNA"), ("UFTN", "RNA"),
    ("UMS", "RNA"), ("UMS5", "RNA"), ("UMS3", "RNA"), ("UMSN", "RNA"),
    ("ADS", "RNA"), ("ADS5", "RNA"), ("ADS3", "RNA"), ("ADSN", "RNA")
]

for res_name, res_type in new_entries:
    clean_lines.append(f"{res_name:<6} {res_type}\n")

with open(local_res_file, "w") as f:
    f.writelines(clean_lines)

print("-> Configured ./residuetypes.dat successfully.")

# ==============================================================================
# 3. ADD NON-BONDED PARAMETERS (PHYSICAL SIGMA IN NANOMETERS)
# ==============================================================================
with open(local_ff / "atomtypes.atp", "a") as f:
    f.write("f         18.99840 ; Fluorine (2'-F ribose)\n")
    f.write("se        78.96000 ; Selenium (2'-SeMe ribose)\n")
    f.write("sp        32.06500 ; Sulfur (Phosphorodithioate P=S)\n")

with open(local_ff / "ffnonbonded.itp", "a") as f:
    f.write("\n; Parameters for modified nucleoside centers (Lorentz-Berthelot comb-rule 2)\n")
    f.write("  f     9    18.99840   0.0000   A   0.31181   0.25522 ; GAFF/Amber Fluorine (sigma = 3.12 A)\n")
    f.write("  se   34    78.96000   0.0000   A   0.36528   1.21754 ; Amber Divalent Se (sigma = 3.65 A)\n")
    f.write("  sp   16    32.06500   0.0000   A   0.35636   1.04600 ; Dithiophosphate S (sigma = 3.56 A)\n")

# ==============================================================================
# 4. ADD BONDED PARAMETERS (INCLUDING WILDCARD DIHEDRALS)
# ==============================================================================
bonded_additions = """
; --- Custom Bonded Parameters for Modified RNA Aptamer ---
[ bondtypes ]
  CT    f    1    0.13800    307105.6
  CT   se    1    0.19700    184100.0
  P    sp    1    0.19500    209200.0

[ angletypes ]
  CT   CT    f    1   109.500    418.400
  f    CT   CT    1   109.500    418.400
  H1   CT    f    1   109.500    418.400
  H2   CT    f    1   109.500    418.400
  CT   CT   se    1   109.500    376.560
  CT   se   CT    1   100.000    334.720
  se   CT   H1    1   109.500    292.880
  H1   CT   se    1   109.500    292.880
  OS    P   sp    1   108.000    376.560
  sp    P   sp    1   116.000    460.240
  O2    P   sp    1   108.000    376.560
  sp    P   O2    1   108.000    376.560

[ dihedraltypes ]
  X    CT   CT    f    9       0.00000       0.00000    1
  f    CT   CT    X    9       0.00000       0.00000    1
  X    CT   se    X    9       0.00000       0.00000    1
  X    se   CT    X    9       0.00000       0.00000    1
  X    OS    P   sp    9       0.00000       0.00000    1
  X    sp    P   OS    9       0.00000       0.00000    1
  X    P    OS   CT    9       0.00000       0.00000    1
"""
with open(local_ff / "ffbonded.itp", "a") as f:
    f.write(bonded_additions)

# ==============================================================================
# 5. REGISTER RESIDUE MAPPING (rna.r2b)
# ==============================================================================
with open(local_ff / "rna.r2b", "a") as f:
    f.write("CFZ    CFZ    CFZ5    CFZ3    CFZN\n")
    f.write("UFT    UFT    UFT5    UFT3    UFTN\n")
    f.write("UMS    UMS    UMS5    UMS3    UMSN\n")
    f.write("ADS    ADS    ADS5    ADS3    ADSN\n")

# ==============================================================================
# 6. CLONE TEMPLATES & CONSTRUCT EXACT .RTP RESIDUE DEFINITIONS
# ==============================================================================
with open(gmx_top / "amber99sb-ildn.ff" / "rna.rtp", "r") as f:
    orig_rtp = f.read()

def parse_rtp_block(rtp_text, resname):
    lines = rtp_text.splitlines()
    in_block = False
    block = []
    for line in lines:
        if line.strip() == f"[ {resname} ]":
            in_block = True
            block.append(line)
            continue
        if in_block:
            if line.strip().startswith("[ ") and not line.strip().startswith(("[ atoms", "[ bonds")):
                break
            block.append(line)
    return "\n".join(block)

rc_block = parse_rtp_block(orig_rtp, "RC")
rc3_block = parse_rtp_block(orig_rtp, "RC3")
ru_block = parse_rtp_block(orig_rtp, "RU")
ra_block = parse_rtp_block(orig_rtp, "RA")

strip_2prime_hydroxyl = {"O2'", "HO'2", "HO2'"}

def create_2prime_fluoro(base_block, new_name):
    lines = base_block.splitlines()
    out = [f"[ {new_name} ]"]
    in_atoms, in_bonds = False, False

    for l in lines[1:]:
        s_line = l.strip()
        if s_line.startswith("[ atoms"):
            in_atoms = True; in_bonds = False; out.append(l); continue
        elif s_line.startswith("[ bonds"):
            in_atoms = False; in_bonds = True; out.append(l); continue

        if in_atoms and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            atom = tokens[0]
            if atom in strip_2prime_hydroxyl:
                continue
            elif atom == "C2'":
                out.append("    C2'        CT       0.1840     5")
                out.append("    F2'        f       -0.3120     5")
                continue
        elif in_bonds and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            if any(h in tokens for h in strip_2prime_hydroxyl):
                continue
        out.append(l)
    out.append("    C2'  F2'")
    return "\n".join(out)

def create_2prime_methylselenyl(ru_block):
    lines = ru_block.splitlines()
    out = ["[ UMS ]"]
    in_atoms, in_bonds = False, False
    for l in lines[1:]:
        s_line = l.strip()
        if s_line.startswith("[ atoms"):
            in_atoms = True; in_bonds = False; out.append(l); continue
        elif s_line.startswith("[ bonds"):
            in_atoms = False; in_bonds = True; out.append(l); continue

        if in_atoms and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            atom = tokens[0]
            if atom in strip_2prime_hydroxyl:
                continue
            elif atom == "C2'":
                out.append("    C2'        CT       0.0679     5")
                out.append("    SE2'       se      -0.2420     5")
                out.append("    CA'        CT      -0.1800     5")
                out.append("    HA'1       H1       0.0754     5")
                out.append("    HA'2       H1       0.0754     5")
                out.append("    HA'3       H1       0.0753     5")
                continue
        elif in_bonds and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            if any(h in tokens for h in strip_2prime_hydroxyl):
                continue
        out.append(l)
    out.extend(["    C2'  SE2'", "    SE2' CA'", "    CA'  HA'1", "    CA'  HA'2", "    CA'  HA'3"])
    return "\n".join(out)

def create_phosphorodithioate(ra_block):
    lines = ra_block.splitlines()
    out = ["[ ADS ]"]
    in_atoms, in_bonds = False, False
    for l in lines[1:]:
        s_line = l.strip()
        if s_line.startswith("[ atoms"):
            in_atoms = True; in_bonds = False; out.append(l); continue
        elif s_line.startswith("[ bonds"):
            in_atoms = False; in_bonds = True; out.append(l); continue

        if in_atoms and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            atom = tokens[0]
            if atom == "P":
                out.append("    P          P        0.8437     1")
                continue
            elif atom in ["O1P", "O2P", "OP1", "OP2"]:
                continue
            elif atom == "O5'":
                out.append("    S2P        sp      -0.6150     1")
                out.append("    S3P        sp      -0.6150     1")
        elif in_bonds and s_line and not s_line.startswith(";"):
            tokens = s_line.split()
            if any(p in tokens for p in ["O1P", "O2P", "OP1", "OP2"]):
                continue
        out.append(l)
    out.extend(["    P    S2P", "    P    S3P"])
    return "\n".join(out)

cfz_rtp = create_2prime_fluoro(rc_block, "CFZ")
uft_rtp = create_2prime_fluoro(ru_block, "UFT")
ums_rtp = create_2prime_methylselenyl(ru_block)
ads_rtp = create_phosphorodithioate(ra_block)
cfz3_rtp = create_2prime_fluoro(rc3_block, "CFZ3")

with open(local_ff / "rna.rtp", "a") as f:
    f.write(f"\n{cfz_rtp}\n{cfz3_rtp}\n{uft_rtp}\n{ums_rtp}\n{ads_rtp}\n")

# ==============================================================================
# 7. APPEND HYDROGEN DATABASE (rna.hdb) WITH H3T FOR CFZ3
# ==============================================================================
hdb_additions = """
CFZ 8
2   6   H5'     C5'     C4'     O5'
1   5   H4'     C4'     C5'     C3'     O4'
1   5   H3'     C3'     C4'     C2'     O3'
1   5   H2'1    C2'     C1'     C3'     F2'
1   5   H1'     C1'     O4'     C2'     N1
2   3   H4      N4      C4      N3
1   1   H5      C5      C4      C6
1   1   H6      C6      C5      N1

CFZ3 9
2   6   H5'     C5'     C4'     O5'
1   5   H4'     C4'     C5'     C3'     O4'
1   5   H3'     C3'     C4'     C2'     O3'
1   5   H2'1    C2'     C1'     C3'     F2'
1   2   H3T     O3'     C3'     C4'
1   5   H1'     C1'     O4'     C2'     N1
2   3   H4      N4      C4      N3
1   1   H5      C5      C4      C6
1   1   H6      C6      C5      N1

UFT 8
2   6   H5'     C5'     C4'     O5'
1   5   H4'     C4'     C5'     C3'     O4'
1   5   H3'     C3'     C4'     C2'     O3'
1   5   H2'1    C2'     C1'     C3'     F2'
1   5   H1'     C1'     O4'     C2'     N1
1   1   H3      N3      C2      C4
1   1   H5      C5      C4      C6
1   1   H6      C6      C5      N1

UMS 9
2   6   H5'     C5'     C4'     O5'
1   5   H4'     C4'     C5'     C3'     O4'
1   5   H3'     C3'     C4'     C2'     O3'
1   5   H2'1    C2'     C1'     C3'     SE2'
3   4   HA'     CA'     SE2'    C2'
1   5   H1'     C1'     O4'     C2'     N1
1   1   H3      N3      C2      C4
1   1   H5      C5      C4      C6
1   1   H6      C6      C5      N1

ADS 9
2   6   H5'     C5'     C4'     O5'
1   5   H4'     C4'     C5'     C3'     O4'
1   5   H3'     C3'     C4'     C2'     O3'
1   5   H2'1    C2'     C1'     C3'     O2'
1   2   HO'2    O2'     C2'     C1'
1   5   H1'     C1'     O4'     C2'     N9
2   3   H6      N6      C6      C5
1   1   H2      C2      N1      N3
1   1   H8      C8      N7      N9
"""
with open(local_ff / "rna.hdb", "a") as f:
    f.write(hdb_additions)

# ==============================================================================
# 8. PRE-PROCESS PDB STRUCTURE (Extract Chain A & Clean Atom Names)
# ==============================================================================
input_pdb = "RNA-Aptamer.pdb"
clean_pdb = "aptamer_clean.pdb"

with open(input_pdb, "r") as fin, open(clean_pdb, "w") as fout:
    for line in fin:
        if line.startswith(("ATOM", "HETATM")):
            chain_id = line[21].strip()
            res_name = line[17:20].strip()

            if chain_id == "A" and res_name in ["A", "U", "G", "C", "CFZ", "UFT", "UMS", "ADS"]:
                line = "ATOM  " + line[6:]
                atom_name = line[12:16].strip()

                if atom_name == "OP1":
                    line = line[:12] + " O1P" + line[16:]
                elif atom_name == "OP2":
                    line = line[:12] + " O2P" + line[16:]
                fout.write(line)
    fout.write("TER\nEND\n")

print(f"-> Generated sanitized structure: {clean_pdb}.")

# ==============================================================================
# 9. RUN PDB2GMX WITH EXPLICIT OUTPUT CAPTURE
# ==============================================================================
pdb2gmx_cmd = [
    "gmx", "pdb2gmx",
    "-f", clean_pdb,
    "-o", "aptamer.gro",
    "-p", "topol.top",
    "-ff", "amber99sb-ildn",
    "-water", "tip3p",
    "-ignh"
]

print("-> Running gmx pdb2gmx...")
proc = subprocess.run(pdb2gmx_cmd, capture_output=True, text=True)

if proc.returncode != 0:
    print(f"FAILED (returncode {proc.returncode}):")
    print(proc.stderr)
else:
    print("SUCCESS: Generated aptamer.gro and topol.top!")

In [ ]:
# @title OPTIONAL - Remove the three 5'-phosphate atoms from truncated DNA sequence
!awk '!(substr($0,1,4)=="ATOM" && substr($0,23,4)+0==6 && substr($0,13,4) ~ /^ *(P|OP1|OP2) *$/)' aptamer_fixed.pdb > aptamer_clean.pdb

!gmx pdb2gmx -f aptamer_clean.pdb -o aptamer.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh

In [ ]:
# @title OPTIONAL - Remove the three 5'-phosphate atoms from truncated RNA sequence
!awk '!($1=="ATOM" && $5=="G" && $6=="1" && ($3=="P" || $3=="OP1" || $3=="OP2"))' aptamer_fixed.pdb > aptamer_clean.pdb

In [ ]:
# @title OPTIONAL - Missing residue in DNA sequence
%%bash
cd /content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD

/usr/local/bin/python - << 'EOF'
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename="aptamer_fixed.pdb")
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.missingTerminals = {}  # Keep DG1 (5') and DC38 (3') termini unchanged for pdb2gmx
print("Rebuilding missing atoms:", fixer.missingAtoms)

fixer.addMissingAtoms()

with open("aptamer_rebuilt.pdb", "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)

# Normalize DC 32 occupancies (0.90 -> 1.00) to clear pdb2gmx occupancy warnings
with open("aptamer_rebuilt.pdb") as f:
    lines = [
        line[:54] + "  1.00" + line[60:] if line.startswith("ATOM") else line
        for line in f
    ]
with open("aptamer_rebuilt.pdb", "w") as f:
    f.writelines(lines)
EOF

gmx pdb2gmx -f aptamer_rebuilt.pdb -o aptamer.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh

In [ ]:
# @title Convert pdb format to gromacs format (gmx) for DNA-Aptamer
!gmx pdb2gmx -f aptamer_fixed.pdb -o aptamer.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh

*System Prepartion*

In [ ]:
# Create box
!gmx editconf -f aptamer.gro -o boxed.gro -c -d 1.0 -bt cubic

In [ ]:
# Solvate the system
!gmx solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top

In [ ]:
# Grompp ion.tpr
!gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1

In [ ]:
# Adding ions into the system
!gmx genion -s ions.tpr -o ionized.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15

*Energy Minimization*

In [ ]:
# Grompp em.tpr
!gmx grompp -f em.mdp -c ionized.gro -p topol.top -o em.tpr -maxwarn 1

In [ ]:
# Minimizing the system
!gmx mdrun -deffnm em -v

*Equilibration*

In [ ]:
# Grompp nvt.tpr
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 1

                :-) GROMACS - gmx grompp, 2026.3-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx
Data prefix:  /usr/local
Working dir:  /content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD
Command line:
  gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 1

Setting the LD random seed to -1073901761

Generated 2346 of the 2346 non-bonded parameter combinations
Generating 1-4 interactions: fudge = 0.5

Generated 2346 of the 2346 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'RNA_chain_A'

turning H bonds into constraints...

Excluding 2 bonded neighbours molecule type 'SOL'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'NA'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'CL'

turning H bonds into constraints...

NOTE 1 [file topol.top, line 7716]:
  System has non-zero total charge: 0.002499
  Total charge should normally be an integer. See
  https://manual.g

In [ ]:
# Equilibrating the system (NVT)
!gmx mdrun -deffnm nvt -v -nb gpu -pme gpu -update cpu

                :-) GROMACS - gmx mdrun, 2026.3-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx
Data prefix:  /usr/local
Working dir:  /content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD
Command line:
  gmx mdrun -deffnm nvt -v -nb gpu -pme gpu -update cpu

Compiled SIMD is AVX2_256, but CPU also supports AVX_512 (see log).
The current CPU can measure timings more accurately than the code in
gmx mdrun was configured to use. This might affect your simulation
speed as accurate timings are needed for load-balancing.
Please consider rebuilding gmx mdrun with the GMX_USE_RDTSCP=ON CMake option.
Reading file nvt.tpr, VERSION 2026.3-conda_forge (single precision)
Changing nstlist from 20 to 100, rlist from 1 to 1.111


1 GPU selected for this run.
Mapping of GPU IDs to the 2 GPU tasks in the 1 rank on this node:
  PP:0,PME:0
PP tasks will do non-perturbed short-ranged interactions on the GPU
PP task will update and constrain coordinates on the CPU
PME tasks will do all aspects on th

In [ ]:
# Grompp npt.tpr
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 1

                :-) GROMACS - gmx grompp, 2026.3-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx
Data prefix:  /usr/local
Working dir:  /content/drive/MyDrive/ColabMD-Edu_Protein-NA/AAMD
Command line:
  gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 1

Setting the LD random seed to 805306363

Generated 2346 of the 2346 non-bonded parameter combinations
Generating 1-4 interactions: fudge = 0.5

Generated 2346 of the 2346 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'RNA_chain_A'

turning H bonds into constraints...

Excluding 2 bonded neighbours molecule type 'SOL'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'NA'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'CL'

turning H bonds into constraints...

NOTE 1 [file topol.top, line 7716]:
  System has non-zero total charge: 0.002499
  Total charge should normally be an integer. See
  https

In [ ]:
# Equilibrating the system (NPT)
!gmx mdrun -deffnm npt -v -nb gpu -pme gpu -update cpu

*Running Simulation*

In [ ]:
# Grompp md.tpr
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 1

In [ ]:
# Running simulation for the system
!gmx mdrun -deffnm md -v -nb gpu -pme gpu -update cpu

*Centered and Fitted Trajectories*

In [ ]:
# Generate xtc file
!gmx trjconv -f md.trr -o md.xtc

In [ ]:
# Center the trajectories
!gmx trjconv -s md.tpr -f md.xtc -o md_noPBC.xtc -pbc mol -center

In [ ]:
# Fit the trajectories
!gmx trjconv -s md.tpr -f md_noPBC.xtc -o md_fit.xtc -fit rot+trans

*Minimized (Relaxed) and Dynamic PDB*

In [ ]:
# Get the minimized pdb
!gmx trjconv -s md.tpr -f md_fit.xtc -o aptamer_relaxed.pdb -dump 1000

In [ ]:
# Convert AA trajectories into PDB format
!gmx trjconv -s md.tpr -f md_noPBC.xtc -dt 100 -o dynamic.pdb